In [1]:
from dotenv import load_dotenv
import json
import requests
import pyodbc
from datetime import datetime
import os

load_dotenv()  # loads .env file
# API_KEY = os.getenv("OWM_API_KEY")
API_KEY = "6f4f6fd9823133fe2823c206c116dd7f"
conn_str = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost\\SQLEXPRESS;"  # double backslash
    "DATABASE=IOT;"
    "UID=docker_user;"
    "PWD=9512"
)
# conn_str = r"DRIVER={ODBC Driver 17 for SQL Server};SERVER=localhost\SQLEXPRESS;DATABASE=IOT;Trusted_Connection=yes;"


In [2]:
print(API_KEY)

6f4f6fd9823133fe2823c206c116dd7f


In [3]:
cities = [
    "Cairo", "Alexandria",
    "New York", "Los Angeles",
    "London", "Manchester",
    "Berlin", "Munich",
    "Paris", "Marseille",
    "Mumbai", "Delhi",
    "Tokyo", "Osaka",
    "Sao Paulo", "Rio de Janeiro",
    "Toronto", "Vancouver",
    "Sydney", "Melbourne"
]

In [7]:
from datetime import datetime, timedelta

# Fake an Airflow execution_date (1 hour ago, for example)
context = {
    "execution_date": datetime.utcnow() - timedelta(hours=1),
    "data_interval_start": datetime.utcnow() - timedelta(hours=1),
    "data_interval_end": datetime.utcnow()
}

C:\Users\husse\AppData\Local\Temp\ipykernel_22000\2425721555.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "execution_date": datetime.utcnow() - timedelta(hours=1),
C:\Users\husse\AppData\Local\Temp\ipykernel_22000\2425721555.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "data_interval_start": datetime.utcnow() - timedelta(hours=1),
C:\Users\husse\AppData\Local\Temp\ipykernel_22000\2425721555.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "data_interval_end": datetime.utcnow()


In [ ]:
conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

for city_name in cities:
    url = f"http://api.openweathermap.org/data/2.5/weather?q={city_name}&appid={API_KEY}&units=metric"
    
    response = requests.get(url, timeout=10)
    if response.status_code != 200:
        print(f"Skipping {city_name}, API error: {response.status_code}")
        continue
    data = response.json()

    # Extract city info
    city_name_api = data.get("name")
    country = data.get("sys", {}).get("country")
    lat = data.get("coord", {}).get("lat")
    lon = data.get("coord", {}).get("lon")

    # Insert city if not exists
    cursor.execute("SELECT CityID FROM Cities WHERE CityName=? AND Country=?", city_name_api, country)
    row = cursor.fetchone()

    if row:
        city_id = row[0]
    else:
        cursor.execute(
            "INSERT INTO Cities (CityName, Country, Latitude, Longitude) OUTPUT INSERTED.CityID VALUES (?, ?, ?, ?)",
            city_name_api, country, lat, lon
        )
        city_id = cursor.fetchone()[0]
    
    conn.commit()

    # Extract weather info
    main = data.get("main", {})
    wind = data.get("wind", {})
    weather_list = data.get("weather", [{}])

    temp = main.get("temp")
    feels_like = main.get("feels_like")
    humidity = main.get("humidity")
    pressure = main.get("pressure")
    wind_speed = wind.get("speed", 0)
    wind_deg = wind.get("deg", 0)
    weather_main = weather_list[0].get("main")
    weather_desc = weather_list[0].get("description")
    now = datetime.now()

    # Insert weather measurement
    cursor.execute("""
        INSERT INTO WeatherMeasurements
        (CityID, Temperature, FeelsLike, Humidity, Pressure, WindSpeed, WindDirection, WeatherMain, WeatherDescription, DateTimeRecorded)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, city_id, temp, feels_like, humidity, pressure, wind_speed, wind_deg, weather_main, weather_desc, now)
    conn.commit()

    print(f"Inserted data for {city_name_api}, {country}")

conn.close()
print("All cities processed successfully!")

History API failed for Cairo (30.0626,31.2497) ts=1759401304: 401 - {"cod":401, "message": "Please note that using One Call 3.0 requires a separate subscription to the One Call by Call plan. Learn more here https://openweathermap.org/price. If you have a valid subscri
History API failed for Alexandria (31.2156,29.9553) ts=1759401304: 401 - {"cod":401, "message": "Please note that using One Call 3.0 requires a separate subscription to the One Call by Call plan. Learn more here https://openweathermap.org/price. If you have a valid subscri
History API failed for New York (40.7143,-74.006) ts=1759401304: 401 - {"cod":401, "message": "Please note that using One Call 3.0 requires a separate subscription to the One Call by Call plan. Learn more here https://openweathermap.org/price. If you have a valid subscri
History API failed for Los Angeles (34.0522,-118.2437) ts=1759401304: 401 - {"cod":401, "message": "Please note that using One Call 3.0 requires a separate subscription to the One Call 